In [1]:
import os
os.chdir('/home/dominik/Downloads')

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import scipy.spatial as spatial

class ElasticBubbleCell:
    def __init__(self, position, radius=0.02):
        self.position = np.array(position, dtype=float)
        self.original_position = np.array(position, dtype=float)  # Reference position
        self.velocity = np.zeros(2, dtype=float)
        self.radius = radius
        self.is_t_cell = False
        
        # More nuanced interaction parameters
        self.t_cell_influence_radius = 0.2
        self.influence_strength = 0.4
        self.position_restoration_factor = 0.05
        self.randomness_factor = 0.001

    def interact(self, all_cells):
        # Find nearby T cells
        nearby_t_cells = [
            cell for cell in all_cells 
            if hasattr(cell, 'is_t_cell') and cell.is_t_cell and 
            np.linalg.norm(cell.nucleus - self.position) < self.t_cell_influence_radius
        ]
        
        # Base velocity computation
        if nearby_t_cells:
            # Weighted influence from T cell movement
            influences = []
            for cell in nearby_t_cells:
                # Distance-based weighting
                dist = np.linalg.norm(cell.nucleus - self.position)
                weight = 1 - (dist / self.t_cell_influence_radius)
                
                # Compute weighted velocity influence
                influence = cell.nucleus_velocity * weight * self.influence_strength
                influences.append(influence)
            
            # Average influences
            if influences:
                self.velocity = np.mean(influences, axis=0)
        else:
            # Minimal random movement and position restoration
            self.velocity = (
                np.random.normal(0, self.randomness_factor, 2) +
                (self.original_position - self.position) * self.position_restoration_factor
            )
        
        # Update position
        self.position += self.velocity
        
        # Soft boundary conditions
        self.position = np.clip(self.position, 0, 1)

class TCell:
    def __init__(self, position):
        # Multi-point representation
        self.nucleus = np.array(position, dtype=float)
        self.leading_edge = self.nucleus + np.random.uniform(-0.03, 0.03, 2)
        self.trailing_edge = self.nucleus - np.random.uniform(-0.03, 0.03, 2)
        
        # Movement parameters
        self.persistence = 0.95
        self.probing_intensity = 0.04
        self.exploration_factor = 0.02
        
        # Add velocity initialization
        self.leading_velocity = np.zeros(2)  # Explicitly initialize leading_velocity
        self.nucleus_velocity = np.zeros(2)
        self.trailing_velocity = np.zeros(2)
    
    def probe_environment(self, bubble_cells):
        """Simulate constrained probing in dense environment"""
        probing_force = np.zeros(2)
        deformation_accumulator = 0
        nearby_threshold = 0.02  # Tight interaction radius
        
        for cell in bubble_cells:
            dist = np.linalg.norm(self.leading_edge - cell.position)
            
            if dist < nearby_threshold:
                # Directional interaction
                direction = (self.leading_edge - cell.position) / (dist + 1e-5)
                
                # Progressive force based on proximity
                force_magnitude = (1 - dist / nearby_threshold) * self.probing_intensity
                probing_force += direction * force_magnitude
                
                # Deformation tracking
                deformation_accumulator += force_magnitude
        
        return probing_force, deformation_accumulator
    
    def move(self, bubble_cells):
        # Probing and force computation
        probing_force, deformation = self.probe_environment(bubble_cells)
        
        # Adaptive exploration with density influence
        exploration = np.random.uniform(-self.exploration_factor, self.exploration_factor, 2)
        
        # Leading edge movement
        self.leading_velocity = (
            self.leading_velocity * self.persistence + 
            probing_force + 
            exploration
        )
        
        # Velocity management
        speed = np.linalg.norm(self.leading_velocity)
        max_speed = 0.015
        if speed > max_speed:
            self.leading_velocity *= max_speed / speed
        
        # Update leading edge
        self.leading_edge += self.leading_velocity
        
        # Nucleus-leading edge dynamics
        nucleus_to_lead_vec = self.leading_edge - self.nucleus
        self.nucleus_velocity = nucleus_to_lead_vec * 0.5
        self.nucleus += self.nucleus_velocity
        
        # Trailing edge update
        self.trailing_velocity = (self.nucleus - self.trailing_edge) * 0.4
        self.trailing_edge += self.trailing_velocity
        
        # Boundary conditions
        for point in [self.nucleus, self.leading_edge, self.trailing_edge]:
            point[:] = np.clip(point, 0, 1)

def create_packed_cell_distribution(
    total_cells=400, 
    packing_density=0.7
):
    """Create a densely packed cell distribution"""
    cells = []
    attempts = 0
    max_attempts = 10000
    
    while len(cells) < total_cells and attempts < max_attempts:
        # Generate candidate position
        candidate = np.random.uniform(0, 1, 2)
        
        # Check overlap with existing cells
        is_valid = True
        for cell in cells:
            dist = np.linalg.norm(candidate - cell.position)
            if dist < packing_density:  # Tight packing radius
                is_valid = False
                break
        
        if is_valid:
            cells.append(ElasticBubbleCell(candidate))
        
        attempts += 1
    
    return cells

def simulate_lymph_node(
    t_cells_count=100, 
    bubble_cells_count=600,
    packing_density=0.7
):
    filename = f'tcell_simulation-t{t_cells_count}-b{bubble_cells_count}-d{packing_density}.mp4'
    
    # Create densely packed bubble cells
    bubble_cells = create_packed_cell_distribution(bubble_cells_count, packing_density)
    
    # Place T cells within this dense environment
    t_cells = [
        TCell(bubble_cells[np.random.randint(len(bubble_cells))].position + 
              np.random.uniform(-0.02, 0.02, 2)) 
        for _ in range(t_cells_count)
    ]
    
    # Visualization setup
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_facecolor('black')
    ax.set_title('T Cell Migration through Dense Cellular Environment', color='white')

    # Scatter plot initialization
    bubble_scatter = ax.scatter([], [], c='lightblue', s=30, alpha=0.3)
    t_cell_nucleus = ax.scatter([], [], c='red', s=50, alpha=0.7)
    t_cell_leading = ax.scatter([], [], c='white', s=30, alpha=0.5)
    t_cell_trailing = ax.scatter([], [], c='blue', s=20, alpha=0.5)
    
    def update(frame):
        # Update bubble cell interactions to maintain density
        for cell in bubble_cells:
            cell.interact(bubble_cells)
        
        # Update T cell movements
        for t_cell in t_cells:
            t_cell.move(bubble_cells)
        
        # Position extraction
        bubble_pos = np.array([cell.position for cell in bubble_cells])
        t_cell_nucleus_pos = np.array([cell.nucleus for cell in t_cells])
        t_cell_leading_pos = np.array([cell.leading_edge for cell in t_cells])
        t_cell_trailing_pos = np.array([cell.trailing_edge for cell in t_cells])
        
        # Update scatter plots
        bubble_scatter.set_offsets(bubble_pos)
        t_cell_nucleus.set_offsets(t_cell_nucleus_pos)
        t_cell_leading.set_offsets(t_cell_leading_pos)
        t_cell_trailing.set_offsets(t_cell_trailing_pos)
        
        return [bubble_scatter, t_cell_nucleus, t_cell_leading, t_cell_trailing]

    # Animation creation
    ani = FuncAnimation(fig, update, frames=300, interval=50, blit=True)
    ani.save(filename, fps=20, dpi=150, writer='ffmpeg')
    plt.close(fig)


In [3]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

class CellTrackingDataGenerator:
    def __init__(self, 
                 t_cells_count=50, 
                 bubble_cells_count=200, 
                 packing_density=0.05,
                 num_simulations=100,
                 grid_size=20,
                 time_steps=50):
        """
        Generate synthetic cell tracking datasets using ABM
        """
        self.t_cells_count = t_cells_count
        self.bubble_cells_count = bubble_cells_count
        self.packing_density = packing_density
        self.num_simulations = num_simulations
        self.grid_size = grid_size
        self.time_steps = time_steps
        
        # Tracking data storage
        self.tracking_data = []
        self.labels = []
    
    def generate_abm_tracking_data(self):
        """
        Generate synthetic tracking data using the ABM simulation
        """
        # Import these here to avoid potential circular import
        # defined in above cell, no need to import for now
        # from your_abm_module import create_packed_cell_distribution, TCell
        
        all_simulation_frames = []
        
        for _ in range(self.num_simulations):
            # Create simulation environment
            bubble_cells = create_packed_cell_distribution(
                total_cells=self.bubble_cells_count, 
                packing_density=self.packing_density
            )
            
            # Place T cells
            t_cells = [
                TCell(bubble_cells[np.random.randint(len(bubble_cells))].position + 
                      np.random.uniform(-0.02, 0.02, 2)) 
                for _ in range(self.t_cells_count)
            ]
            
            # Collect frames of cell positions
            simulation_frames = []
            for _ in range(50):  # 50 time steps
                # Update bubble cells and T cells
                for cell in bubble_cells:
                    cell.interact(bubble_cells)
                
                for t_cell in t_cells:
                    t_cell.move(bubble_cells)
                
                # Extract positions
                frame = {
                    'bubble_cells': np.array([cell.position for cell in bubble_cells]),
                    't_cell_nuclei': np.array([cell.nucleus for cell in t_cells]),
                    't_cell_leading': np.array([cell.leading_edge for cell in t_cells]),
                    't_cell_trailing': np.array([cell.trailing_edge for cell in t_cells])
                }
                simulation_frames.append(frame)
            
            all_simulation_frames.append(simulation_frames)
        
        return all_simulation_frames

    def prepare_ml_dataset(self):
        simulation_data = self.generate_abm_tracking_data()
        
        X = []  # Input features
        y = []  # Target trajectories
        
        # Collect ALL positions to get truly global normalization
        all_positions = []
        for simulation in simulation_data:
            for frame in simulation:
                all_positions.extend(frame['t_cell_nuclei'])
        
        # Global normalization parameters
        global_min = np.min(all_positions, axis=0)
        global_max = np.max(all_positions, axis=0)
        
        for simulation in simulation_data:
            sim_frames = []
            sim_trajectories = []
            
            for frame in simulation:
                # IMPROVED Normalization
                def robust_normalize(positions):
                    if len(positions) == 0:
                        return np.zeros((self.t_cells_count, 2))
                    
                    # Ensure consistent shape WITHOUT zero-padding
                    positions = positions[:self.t_cells_count]
                    
                    # Robust normalization across ALL simulations
                    normalized = (positions - global_min) / (global_max - global_min)
                    
                    # Handle potential edge cases
                    normalized = np.clip(normalized, 0, 1)
                    
                    return normalized
                
                # Create grid representation
                grid = np.zeros((self.grid_size, self.grid_size, 4), dtype=np.float32)
                
                # Use robust normalization
                grid[:,:,0] = self._create_density_map(robust_normalize(frame['bubble_cells']))
                grid[:,:,1] = self._create_density_map(robust_normalize(frame['t_cell_nuclei']))
                grid[:,:,2] = self._create_density_map(robust_normalize(frame['t_cell_leading']))
                grid[:,:,3] = self._create_density_map(robust_normalize(frame['t_cell_trailing']))
                
                sim_frames.append(grid)
                
                # Store normalized trajectories
                sim_trajectories.append(robust_normalize(frame['t_cell_nuclei']))
            
            X.append(sim_frames)
            y.append(np.array(sim_trajectories))
        
        # Convert to numpy arrays
        X = np.array(X)
        y = np.array(y)
        
        # Detailed debugging
        print("Simulation Data Overview:")
        print("Total Simulations:", len(simulation_data))
        print("X shape:", X.shape)
        print("y shape:", y.shape)
        print("\nGlobal Normalization:")
        print("Global Min:", global_min)
        print("Global Max:", global_max)

        # plot out
        self.visualize_raw_tracks(simulation_data)
        
        # Optional: More rigorous sanity checks
        self._validate_dataset(X, y)
        
        return X, y
    
    def _create_density_map(self, normalized_positions):
        """
        Create a smoother density representation
        """
        grid_map = np.zeros((self.grid_size, self.grid_size))
        
        # Use Gaussian-like distribution instead of discrete mapping
        for pos in normalized_positions:
            x = pos[0] * (self.grid_size - 1)
            y = pos[1] * (self.grid_size - 1)
            
            # Spread influence around the point
            x_int, y_int = int(x), int(y)
            
            # Simple weighted distribution
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    weight = 1 - np.sqrt(dx**2 + dy**2) / np.sqrt(2)
                    if 0 <= x_int+dx < self.grid_size and 0 <= y_int+dy < self.grid_size:
                        grid_map[x_int+dx, y_int+dy] += weight
        
        # Normalize
        return grid_map / (np.max(grid_map) if np.max(grid_map) > 0 else 1)
    
    def _validate_dataset(self, X, y):
        """
        Additional sanity checks for the dataset
        """
        # Check for NaN or infinite values
        assert not np.isnan(X).any(), "Dataset contains NaN values"
        assert not np.isnan(y).any(), "Trajectory contains NaN values"
        
        # Check value ranges
        assert np.all((X >= 0) & (X <= 1)), "X values out of [0,1] range"
        assert np.all((y >= 0) & (y <= 1)), "Trajectory values out of [0,1] range"
        
        # Check trajectory consistency
        trajectory_variations = np.std(y, axis=1)
        print("\nTrajectory Variation Statistics:")
        print("Mean Variation:", np.mean(trajectory_variations))
        print("Max Variation:", np.max(trajectory_variations))
    
    def visualize_raw_tracks(self, simulation_data):
        """
        Visualize raw tracks from the ABM simulation
        """
        plt.figure(figsize=(15, 10))
        
        # Plot trajectories for multiple simulations
        for sim_idx in range(min(5, len(simulation_data))):
            plt.subplot(2, 3, sim_idx + 1)
            
            # Extract T cell nucleus positions for this simulation
            simulation = simulation_data[sim_idx]
            t_cell_tracks = []
            
            # Collect T cell nucleus positions for each frame
            for frame in simulation:
                t_cell_positions = frame['t_cell_nuclei']
                t_cell_tracks.append(t_cell_positions)
            
            # Convert to numpy array for easier manipulation
            t_cell_tracks = np.array(t_cell_tracks)
            
            # Plot each T cell's trajectory
            for cell_idx in range(t_cell_tracks.shape[1]):
                plt.plot(
                    t_cell_tracks[:, cell_idx, 0], 
                    t_cell_tracks[:, cell_idx, 1], 
                    label=f'Cell {cell_idx}'
                )
            
            plt.title(f'Simulation {sim_idx} T Cell Tracks')
            plt.xlabel('X Position')
            plt.ylabel('Y Position')
        
        plt.tight_layout()
        plt.savefig('model_outputs/raw_cell_tracks.png')
        plt.close()


2026-02-19 11:36:23.529990: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-19 11:36:23.800743: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-19 11:36:25.059576: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
class MemoryEfficientMultiCellTrackingModel(nn.Module):
    def __init__(self, input_shape, num_cells=50, time_steps=50):
        super().__init__()
        
        time_dim, height, width, channels = input_shape
        
        # Reduced feature extractor with fewer parameters
        self.feature_extractor = nn.Sequential(
            nn.Conv3d(channels, 16, kernel_size=(3, 3, 3), padding=1, stride=2),
            nn.BatchNorm3d(16),
            nn.LeakyReLU(0.2),
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), padding=1, stride=2),
            nn.BatchNorm3d(32),
            nn.LeakyReLU(0.2)
        )
        
        # Dynamically calculate flattened size
        self.to('cuda')  # Move the model to the GPU
        self.half()  # Cast the model to half precision
        
        with torch.no_grad():
            dummy_input = torch.zeros(1, channels, time_dim, height, width, device='cuda', dtype=torch.float16)
            features = self.feature_extractor(dummy_input)
            self.feature_size = features.numel() // features.size(0)
        
        # Lighter feature processor
        self.shared_processor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.feature_size, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3)
        )
        
        # Lightweight cell predictors
        self.cell_predictors = nn.ModuleList([
            nn.Sequential(
                nn.Linear(256, 64),
                nn.BatchNorm1d(64),
                nn.LeakyReLU(0.2),
                nn.Linear(64, time_steps * 2)
            ) for _ in range(num_cells)
        ])
        
        self.num_cells = num_cells
        self.time_steps = time_steps
    
    def forward(self, x):
        # Reduce memory by processing in smaller chunks if needed
        batch_size = x.size(0)
        cell_tracks = []
        
        # Process in smaller batches to reduce memory pressure
        for start in range(0, batch_size, 4):  # Adjust batch size as needed
            end = min(start + 4, batch_size)
            batch_x = x[start:end]
            
            # Move to GPU only when processing
            batch_x = batch_x.to(self.feature_extractor[0].weight.device)
            
            batch_x = batch_x.permute(0, 4, 1, 2, 3)
            features = self.feature_extractor(batch_x)
            shared_features = self.shared_processor(features)
            
            batch_cell_tracks = []
            for predictor in self.cell_predictors:
                cell_track = predictor(shared_features)
                cell_track = cell_track.view(-1, self.time_steps, 2)
                batch_cell_tracks.append(cell_track)
            
            batch_cell_tracks = torch.stack(batch_cell_tracks, dim=1)
            cell_tracks.append(batch_cell_tracks)
            
            # Clear cuda cache periodically
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        return torch.cat(cell_tracks, dim=0)

class TrainingPipeline:
    def __init__(self, X, num_cells=50):
        # Use lower precision to reduce memory
        torch.set_default_dtype(torch.float16)
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        input_shape = (X.shape[1], X.shape[2], X.shape[3], X.shape[4])
        
        self.model = MemoryEfficientMultiCellTrackingModel(
            input_shape=input_shape, 
            num_cells=num_cells,
            time_steps=X.shape[1]
        ).to(self.device).half()
        
        # Use more memory-efficient optimizer
        self.optimizer = optim.AdamW(
            self.model.parameters(), 
            lr=1e-3, 
            weight_decay=1e-5,
            foreach=True  # More efficient parameter updates
        )
        
        # Lighter scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, 
            T_max=50  # Adjust based on your epochs
        )
    
    def multi_cell_trajectory_loss(self, predictions, targets):
        # Compute loss in lower precision
        with torch.cuda.amp.autocast():
            distance_loss = F.mse_loss(predictions, targets)
            
            pred_diff = predictions[:, :, 1:] - predictions[:, :, :-1]
            target_diff = targets[:, :, 1:] - targets[:, :, :-1]
            continuity_loss = F.mse_loss(pred_diff, target_diff)
            
            total_loss = distance_loss + 0.1 * continuity_loss
        
        return total_loss
    
    def train(self, X, y, epochs=100, batch_size=8, patience=15):
        # Use mixed precision training
        scaler = torch.cuda.amp.GradScaler()
        
        # Convert to PyTorch tensors and lower precision
        X = torch.from_numpy(X).to(self.device).half()
        y = torch.from_numpy(y).to(self.device).half()
        
        X_train, X_test, y_train, y_test = train_test_split(
            X.cpu().numpy(), y.cpu().numpy(), test_size=0.2, random_state=42
        )
        
        # Convert back to PyTorch tensors and lower precision
        X_train = torch.from_numpy(X_train).to(self.device).half()
        X_test = torch.from_numpy(X_test).to(self.device).half()
        y_train = torch.from_numpy(y_train).to(self.device).half()
        y_test = torch.from_numpy(y_test).to(self.device).half()
        
        training_history = {
            'train_loss': [],
            'val_loss': []
        }
        
        best_val_loss = float('inf')
        epochs_no_improve = 0
        
        for epoch in range(epochs):
            self.model.train()
            total_train_loss = 0
            
            for i in range(0, len(X_train), batch_size):
                batch_X = X_train[i:i+batch_size]
                batch_y = y_train[i:i+batch_size]
                
                self.optimizer.zero_grad()
                
                with torch.cuda.amp.autocast():
                    predictions = self.model(batch_X)
                    loss = self.multi_cell_trajectory_loss(predictions, batch_y)
                
                loss.backward()
                self.optimizer.step()
                
                total_train_loss += loss.item()
                
                # Clear cuda cache
                torch.cuda.empty_cache()
            
            self.scheduler.step()
            
            # Validation with reduced precision
            self.model.eval()
            with torch.no_grad(), torch.cuda.amp.autocast():
                val_predictions = self.model(X_test)
                val_loss = self.multi_cell_trajectory_loss(val_predictions, y_test)
            
            avg_train_loss = total_train_loss / len(X_train)
            training_history['train_loss'].append(avg_train_loss)
            training_history['val_loss'].append(val_loss.item())
            
            print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {val_loss.item():.4f}")
            
            # Early stopping logic
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                torch.save(self.model.state_dict(), 'best_multi_cell_model.pth')
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break
        
        return self.model, training_history

    def visualize_tracks(self, X_test, y_test):
        """
        Comprehensive track visualization
        """
        # Ensure model is in evaluation mode
        self.model.eval()
        
        # Convert to tensors
        X_test = torch.FloatTensor(X_test).to(self.device)
        y_test = torch.FloatTensor(y_test).to(self.device)
        
        # Predict tracks
        with torch.no_grad():
            predicted_tracks = self.model(X_test).cpu().numpy()
            y_test = y_test.cpu().numpy()
        
        # Visualization
        plt.figure(figsize=(20, 15))
        
        # Plot a subset of original and predicted tracks
        num_cells_to_plot = min(10, predicted_tracks.shape[1])
        
        # Original Tracks
        plt.subplot(2, 2, 1)
        for cell_idx in range(num_cells_to_plot):
            plt.plot(y_test[0, cell_idx, :, 0], y_test[0, cell_idx, :, 1], 
                     label=f'Original Cell {cell_idx}')
        plt.title('Original Cell Tracks')
        plt.xlabel('X Coordinate')
        plt.ylabel('Y Coordinate')
        plt.legend()
        
        # Predicted Tracks
        plt.subplot(2, 2, 2)
        for cell_idx in range(num_cells_to_plot):
            plt.plot(predicted_tracks[0, cell_idx, :, 0], 
                     predicted_tracks[0, cell_idx, :, 1], 
                     label=f'Predicted Cell {cell_idx}')
        plt.title('Predicted Cell Tracks')
        plt.xlabel('X Coordinate')
        plt.ylabel('Y Coordinate')
        plt.legend()
        
        # Track Comparison
        plt.subplot(2, 2, 3)
        for cell_idx in range(num_cells_to_plot):
            plt.scatter(y_test[0, cell_idx, :, 0], 
                        y_test[0, cell_idx, :, 1], 
                        c='blue', alpha=0.5, label=f'Original Cell {cell_idx}')
            plt.scatter(predicted_tracks[0, cell_idx, :, 0], 
                        predicted_tracks[0, cell_idx, :, 1], 
                        c='red', alpha=0.5, label=f'Predicted Cell {cell_idx}')
        plt.title('Original vs Predicted Tracks')
        plt.xlabel('X Coordinate')
        plt.ylabel('Y Coordinate')
        plt.legend()
        
        # Error Distribution
        plt.subplot(2, 2, 4)
        errors = y_test[0] - predicted_tracks[0]
        plt.hist(errors.flatten(), bins=50)
        plt.title('Prediction Error Distribution')
        plt.xlabel('Error Magnitude')
        plt.ylabel('Frequency')
        
        plt.tight_layout()
        plt.savefig('model_outputs/multi_cell_tracking_analysis.png')
        plt.close()
        
        # Print some diagnostic information
        print("\nMulti-Cell Tracking Diagnostics:")
        print(f"Total Cells Tracked: {predicted_tracks.shape[1]}")
        print(f"Time Steps: {predicted_tracks.shape[2]}")
        print(f"Coordinates per Cell: {predicted_tracks.shape[3]}")
        
        return predicted_tracks, y_test

# Main execution function
# Use your existing data generation code
data_generator = CellTrackingDataGenerator(
    t_cells_count=50,
    bubble_cells_count=200,
    packing_density=0.05,
    num_simulations=12
)

# Prepare dataset
X, y = data_generator.prepare_ml_dataset()

# Print data shapes for verification
print("Input Data Shapes:")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Initialize training pipeline
training_pipeline = TrainingPipeline(X, num_cells=X.shape[1])

# Train the model
trained_model, training_history = training_pipeline.train(X, y)

# Visualize tracks
predicted_tracks, true_tracks = training_pipeline.visualize_tracks(X[:10], y[:10])

# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(training_history['train_loss'], label='Training Loss')
plt.plot(training_history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig('training_history.png')
plt.close()

Simulation Data Overview:
Total Simulations: 12
X shape: (12, 50, 20, 20, 4)
y shape: (12, 50, 50, 2)

Global Normalization:
Global Min: [0. 0.]
Global Max: [1. 1.]

Trajectory Variation Statistics:
Mean Variation: 0.04108222627154187
Max Variation: 0.12210976158023318
Input Data Shapes:
X shape: (12, 50, 20, 20, 4)
y shape: (12, 50, 50, 2)


/tmp/ipykernel_17721/2022989564.py:129: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_17721/2022989564.py:164: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_17721/2022989564.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


ValueError: Attempting to unscale FP16 gradients.

In [ ]:
tracking_model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

class CellTrackPredictionEvaluator:
    def __init__(self, trained_model, grid_size=20):
        """
        Initialize prediction evaluator
        
        Parameters:
        - trained_model: Trained Keras model for T cell tracking
        - grid_size: Size of the grid used in training
        """
        self.model = trained_model
        self.grid_size = grid_size
    
    def prepare_abm_data_for_prediction(self, t_cells, bubble_cells, num_frames=50):
        """
        Convert ABM simulation data to model input format
        
        Key Change: Capture initial positions explicitly
        """
        # Store initial positions before any movement
        initial_positions = np.array([cell.nucleus for cell in t_cells])
        
        # Create input tensor
        X = np.zeros((1, num_frames, self.grid_size, self.grid_size, 4), dtype=np.float32)
        
        # Helper function to map positions to grid
        def map_to_grid(positions, channel):
            # Scale positions to grid coordinates
            scaled_pos = (np.array(positions) * self.grid_size).astype(int)
            scaled_pos = np.clip(scaled_pos, 0, self.grid_size-1)
            
            # Mark positions on specific channel
            for x, y in scaled_pos:
                X[0, frame, x, y, channel] = 1.0
        
        # Simulate and record frames
        for frame in range(num_frames):
            # Update bubble cell interactions
            for cell in bubble_cells:
                cell.interact(bubble_cells)
            
            # Update T cell movements
            for t_cell in t_cells:
                t_cell.move(bubble_cells)
            
            # Map cell positions to grid
            map_to_grid([cell.position for cell in bubble_cells], 0)      # Bubble cells
            map_to_grid([cell.nucleus for cell in t_cells], 1)            # T cell nuclei
            map_to_grid([cell.leading_edge for cell in t_cells], 2)       # Leading edges
            map_to_grid([cell.trailing_edge for cell in t_cells], 3)      # Trailing edges
        
        return X, initial_positions
    
    def extract_true_tracks(self, t_cells, num_frames=50):
        """
        Extract true T cell tracks from ABM simulation
        
        Parameters:
        - t_cells: List of T cells from ABM simulation
        - num_frames: Number of frames to process
        
        Returns:
        - True tracks for each T cell
        """
        # Track nuclei positions over time
        true_tracks = []
        for t_cell in t_cells:
            cell_track = []
            for _ in range(num_frames):
                cell_track.append(t_cell.nucleus.copy())
                t_cell.move([])  # Move without bubble cells for track extraction
            true_tracks.append(cell_track)
        
        return np.array(true_tracks)
    
    def evaluate_tracking_performance(self, true_tracks, predicted_tracks):
        """
        Compute detailed tracking performance metrics
        
        Parameters:
        - true_tracks: Original T cell tracks from ABM
        - predicted_tracks: Model-predicted tracks
        
        Returns:
        - Dictionary of performance metrics
        """
        # Reshape tracks for comparison
        true_tracks_reshaped = true_tracks.reshape(true_tracks.shape[0], -1, 2)
        predicted_tracks_reshaped = predicted_tracks.reshape(predicted_tracks.shape[0], -1, 2)
        
        # Compute metrics
        metrics = {
            'Mean Absolute Error (MAE)': np.mean(np.abs(true_tracks_reshaped - predicted_tracks_reshaped)),
            'Root Mean Squared Error (RMSE)': np.sqrt(np.mean(np.square(true_tracks_reshaped - predicted_tracks_reshaped))),
            'Mean Tracking Error': np.mean(np.linalg.norm(true_tracks_reshaped - predicted_tracks_reshaped, axis=2)),
            'Tracking Consistency': np.mean(np.all(np.abs(true_tracks_reshaped - predicted_tracks_reshaped) < 0.05, axis=(1,2)))
        }
        
        return metrics
    
    def visualize_tracks(self, true_tracks, predicted_tracks, num_cells=5):
        """
        Create comparative visualization of true and predicted tracks
        
        Parameters:
        - true_tracks: Original T cell tracks from ABM
        - predicted_tracks: Model-predicted tracks
        - num_cells: Number of cells to visualize
        """
        plt.figure(figsize=(15, 6))
        
        # True tracks subplot
        plt.subplot(1, 2, 1)
        true_tracks_reshaped = true_tracks.reshape(true_tracks.shape[0], -1, 2)
        for i in range(min(num_cells, true_tracks_reshaped.shape[0])):
            plt.plot(true_tracks_reshaped[i, :, 0], true_tracks_reshaped[i, :, 1], 
                     label=f'True Track {i+1}', marker='o', markersize=3)
        plt.title('True T Cell Tracks')
        plt.xlabel('X Position')
        plt.ylabel('Y Position')
        plt.legend()
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        
        # Predicted tracks subplot
        plt.subplot(1, 2, 2)
        predicted_tracks_reshaped = predicted_tracks.reshape(predicted_tracks.shape[0], -1, 2)
        for i in range(min(num_cells, predicted_tracks_reshaped.shape[0])):
            plt.plot(predicted_tracks_reshaped[i, :, 0], predicted_tracks_reshaped[i, :, 1], 
                     label=f'Predicted Track {i+1}', marker='o', markersize=3)
        plt.title('Predicted T Cell Tracks')
        plt.xlabel('X Position')
        plt.ylabel('Y Position')
        plt.legend()
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        
        plt.tight_layout()
        plt.savefig('model_outputs/t_cell_track_comparison.png')
        plt.close()

def main():
    # Set random seeds for reproducibility
    np.random.seed(42)
    tf.random.set_seed(42)
    
    # Create ABM simulation
    # Adjust parameters as needed
    t_cells_count = 50
    bubble_cells_count = 200
    packing_density = 0.05
    
    # Create bubble cells and T cells
    bubble_cells = create_packed_cell_distribution(
        total_cells=bubble_cells_count, 
        packing_density=packing_density
    )
    
    # Place T cells
    t_cells = [
        TCell(bubble_cells[np.random.randint(len(bubble_cells))].position + 
              np.random.uniform(-0.02, 0.02, 2)) 
        for _ in range(t_cells_count)
    ]
    
    # Initialize prediction evaluator
    track_evaluator = CellTrackPredictionEvaluator(tracking_model.model)
    
    # Prepare input data for prediction
    X_test, initial_positions = track_evaluator.prepare_abm_data_for_prediction(t_cells, bubble_cells)
    
    # Extract true tracks
    true_tracks = track_evaluator.extract_true_tracks(t_cells)
    
    # Predict tracks
    predicted_tracks = track_evaluator.model.predict(X_test)
    
    # Reshape predicted tracks
    predicted_tracks = predicted_tracks.reshape(true_tracks.shape)
    
    # Align starting positions
    for i in range(predicted_tracks.shape[0]):
        predicted_tracks[i, 0, :] = initial_positions[i]
    
    # Visualize tracks
    track_evaluator.visualize_tracks(true_tracks, predicted_tracks)
    
    # Evaluate performance
    performance_metrics = track_evaluator.evaluate_tracking_performance(true_tracks, predicted_tracks)
    
    # Print performance metrics
    print("\nTracking Performance Metrics:")
    for metric, value in performance_metrics.items():
        print(f"{metric}: {value}")

# Run the main function
if __name__ == '__main__':
    main()
